In [1]:
# This notebook evaluates trained models on a single 80/20 test split using wealth-inclusive features
import numpy as np
import pandas as pd

df = pd.read_parquet("INDIA/DATA/6.IN_TEST_20SET.parquet", engine="pyarrow")

In [ ]:
#normalize to numeric first
cols_to_numeric = [
    "hv024RegionDivision",
    "hv025TypeOfResidence",
    "hv219SexOfHead",
    "hv270WealthIndex",  
    "hv106_01EducationOfHead",
    "hv115_01MaritalStatus",
    "v717Occupation",
    "745aHouseOwnership",
]
for col in cols_to_numeric:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"   
    )

In [2]:
def restore_ml_dtypes(df):
    df = df.copy()

    df["hv220AgeOfHead"] = df["hv220AgeOfHead"].astype("int8")
    df["hv009FamilySize"] = df["hv009FamilySize"].astype("int8")
    df["hv216HouseSize"] = df["hv216HouseSize"].astype("float64")
    df["hv014NoOfChildren"] = df["hv014NoOfChildren"].astype("int8")

    df[[
        "hv024RegionDivision",
        "hv025TypeOfResidence",
        "hv219SexOfHead",
        "hv270WealthIndex",
        "hv106_01EducationOfHead",
        "hv115_01MaritalStatus",
        "v717Occupation",
        "745aHouseOwnership",
        "hv247HasBankAccount",
        "v714CurrentlyWorking",
    ]] = df[[
        "hv024RegionDivision",
        "hv025TypeOfResidence",
        "hv219SexOfHead",
        "hv270WealthIndex",
        "hv106_01EducationOfHead",
        "hv115_01MaritalStatus",
        "v717Occupation",
        "745aHouseOwnership",
        "hv247HasBankAccount",
        "v714CurrentlyWorking",
    ]].astype("category")

    df[[
        "EnergyPoor",
    ]] = df[[
        "EnergyPoor",
    ]].astype("int8")

    return df

In [3]:
df_restore = restore_ml_dtypes(df)

In [4]:
print("\nDtypes:")
print(df_restore.dtypes)

print("\nMissing values:")
print(df_restore.isna().sum().sort_values(ascending=False).head())


Dtypes:
hv270WealthIndex             category
hv024RegionDivision          category
hv025TypeOfResidence         category
hv219SexOfHead               category
hv220AgeOfHead                   int8
hv106_01EducationOfHead      category
hv115_01MaritalStatus        category
hv009FamilySize                  int8
hv247HasBankAccount          category
hv216HouseSize                   int8
hv014NoOfChildren                int8
v714CurrentlyWorking         category
v717Occupation               category
745aHouseOwnership           category
HouseholdClusterElevation     float64
EnergyPoor                       int8
dtype: object

Missing values:
hv270WealthIndex        0
hv024RegionDivision     0
hv025TypeOfResidence    0
hv219SexOfHead          0
hv220AgeOfHead          0
dtype: int64


In [5]:
TARGET = "EnergyPoor"

drop_cols_mepi_wealth_inclusive = [

    # Country (not needed for per-country models)
    "hv000CountryCode",

    # ID variables
    "hhidCaseIdentification",
    "hv001ClusterNumber",
    "hv002HouseholdNumber",

    # Sampling weights
    "hv005SimplilingWeight",
    "weight",
]

# Build clean training dataframe
X_cols = [c for c in df_restore.columns
          if c not in drop_cols_mepi_wealth_inclusive + [TARGET]]

test_df = df_restore[X_cols + [TARGET]]
print("WEALTH-INCLUSIVE SPECIFICATION")
print("Target:", TARGET)
print("Number of features:", len(X_cols))
print("Features used:", X_cols)

WEALTH-INCLUSIVE SPECIFICATION
Target: EnergyPoor
Number of features: 15
Features used: ['hv270WealthIndex', 'hv024RegionDivision', 'hv025TypeOfResidence', 'hv219SexOfHead', 'hv220AgeOfHead', 'hv106_01EducationOfHead', 'hv115_01MaritalStatus', 'hv009FamilySize', 'hv247HasBankAccount', 'hv216HouseSize', 'hv014NoOfChildren', 'v714CurrentlyWorking', 'v717Occupation', '745aHouseOwnership', 'HouseholdClusterElevation']


In [6]:
print("\nDtypes:")
print(test_df.dtypes)

print("\nMissing values:")
print(test_df.isna().sum().sort_values(ascending=False).head())


Dtypes:
hv270WealthIndex             category
hv024RegionDivision          category
hv025TypeOfResidence         category
hv219SexOfHead               category
hv220AgeOfHead                   int8
hv106_01EducationOfHead      category
hv115_01MaritalStatus        category
hv009FamilySize                  int8
hv247HasBankAccount          category
hv216HouseSize                   int8
hv014NoOfChildren                int8
v714CurrentlyWorking         category
v717Occupation               category
745aHouseOwnership           category
HouseholdClusterElevation     float64
EnergyPoor                       int8
dtype: object

Missing values:
hv270WealthIndex        0
hv024RegionDivision     0
hv025TypeOfResidence    0
hv219SexOfHead          0
hv220AgeOfHead          0
dtype: int64


In [ ]:
import pandas as pd
import numpy as np
from autogluon.tabular import TabularPredictor
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, matthews_corrcoef

# --- Paths ---
MODEL_PATH = "INDIA/WEALTH_INCLUSIVE/TRADITIONAL/"
DL_MODEL_PATH = "INDIA/WEALTH_INCLUSIVE/DL/"
DLFT_MODEL_PATH = "INDIA/WEALTH_INCLUSIVE/DLFT/"


# --- Load predictors ---
predictor_tr = TabularPredictor.load(MODEL_PATH)
predictor_dl = TabularPredictor.load(DL_MODEL_PATH)
predictor_ft = TabularPredictor.load(DLFT_MODEL_PATH)

LABEL = "EnergyPoor"

# -----------------------------
# chunk-safe metrics (unchanged)
# -----------------------------
def safe_binary_metrics(predictor, df, label=LABEL, chunk_size=20000, pos_label=None):
    # Decide positive label automatically if not given
    if pos_label is None:
        uniq = list(np.unique(df[label].dropna().to_numpy()))
        if 1 in uniq:
            pos_label = 1
        elif "1" in uniq:
            pos_label = "1"
        else:
            try:
                pos_label = max(uniq)
            except Exception:
                pos_label = uniq[-1]

    TP = TN = FP = FN = 0
    n = len(df)

    for start in range(0, n, chunk_size):
        chunk = df.iloc[start:start + chunk_size]
        y_true = chunk[label].to_numpy()
        y_pred = predictor.predict(chunk).to_numpy()

        yt = (y_true == pos_label)
        yp = (y_pred == pos_label)

        TP += int(np.sum(yp & yt))
        TN += int(np.sum(~yp & ~yt))
        FP += int(np.sum(yp & ~yt))
        FN += int(np.sum(~yp & yt))

    total = TP + TN + FP + FN

    accuracy = (TP + TN) / total if total else 0.0
    tpr = TP / (TP + FN) if (TP + FN) else 0.0
    tnr = TN / (TN + FP) if (TN + FP) else 0.0
    balanced_accuracy = 0.5 * (tpr + tnr)
    f1 = (2 * TP) / (2 * TP + FP + FN) if (2 * TP + FP + FN) else 0.0

    denom = (TP + FP) * (TP + FN) * (TN + FP) * (TN + FN)
    mcc = ((TP * TN - FP * FN) / np.sqrt(denom)) if denom else 0.0

    return {
        "pos_label": pos_label,
        "TP": TP, "TN": TN, "FP": FP, "FN": FN,
        "Accuracy": accuracy,
        "Balanced_Accuracy": balanced_accuracy,
        "F1": f1,
        "MCC": mcc,
    }

# --- Function for Traditional + DL (leaderboard) ---
def get_formatted_results(predictor, test_df, country_label):
    lb = predictor.leaderboard(
        data=test_df,
        extra_metrics=["f1", "mcc", "balanced_accuracy"],
        silent=True
    )

    results = (
        lb[[
            "model",
            "score_test",
            "balanced_accuracy",
            "f1",
            "mcc"
        ]]
        .rename(columns={
            "model": "Model",
            "score_test": "Accuracy",
            "balanced_accuracy": "Balanced Accuracy",
            "f1": "F1",
            "mcc": "MCC"
        })
    )

    results.insert(0, "Country", country_label)
    return results

# --- Function for FT (uses safe_binary_metrics + chunk logic) ---
def get_ft_results(
    predictor,
    test_df,
    country_label,
    chunk_threshold=50000,
    chunk_size=20000,
    pos_label=1
):
    # Get model name safely
    model_name = predictor.leaderboard(silent=True)["model"].iloc[0]

    # Large dataset -> chunk metrics
    if len(test_df) >= chunk_threshold:
        metrics = safe_binary_metrics(
            predictor,
            test_df,
            label=LABEL,
            chunk_size=chunk_size,
            pos_label=pos_label
        )
        acc = metrics["Accuracy"]
        bal_acc = metrics["Balanced_Accuracy"]
        f1 = metrics["F1"]
        mcc = metrics["MCC"]

    # Small dataset -> normal metrics
    else:
        y_true = test_df[LABEL]
        y_pred = predictor.predict(test_df, model=model_name)

        acc = accuracy_score(y_true, y_pred)
        bal_acc = balanced_accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, pos_label=pos_label)
        mcc = matthews_corrcoef(y_true, y_pred)

    results = pd.DataFrame([{
        "Country": country_label,
        "Model": model_name,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "F1": f1,
        "MCC": mcc
    }])

    return results

COUNTRY = "India_Wealth_Inclusive "


# --- Create results from 3 predictors ---
res_tr = get_formatted_results(predictor_tr, test_df, COUNTRY)
res_dl = get_formatted_results(predictor_dl, test_df, COUNTRY)
res_ft = get_ft_results(predictor_ft, test_df, COUNTRY)  # auto uses chunking only if needed

# --- Combine all (8 models total) ---
results_all = pd.concat([res_tr, res_dl, res_ft], ignore_index=True)

# --- Remove ensemble rows if any (safety) ---
results_all = results_all[
    ~results_all["Model"].str.contains("WeightedEnsemble", na=False)
]

# --- Sort by Accuracy highest first ---
results_all = results_all.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

# --- Round to 3 decimals ---
metric_cols = ["Accuracy", "Balanced Accuracy", "F1", "MCC"]
results_all[metric_cols] = results_all[metric_cols].astype(float).round(3)

results_all

In [13]:
results_all.to_csv("INDIA/RESULT/TableA1_india_model_performance_wealth_inclusive.csv", index=False)

In [ ]:
import pandas as pd

# --- Leaderboards for Traditional + DL ---
lb_tr = predictor_tr.leaderboard(
    data=test_df,
    extra_metrics=["f1", "mcc", "balanced_accuracy"],
    silent=True
)

lb_dl = predictor_dl.leaderboard(
    data=test_df,
    extra_metrics=["f1", "mcc", "balanced_accuracy"],
    silent=True
)

# --- Candidates from Traditional + DL ---
cand_tr = lb_tr[["model", "score_test"]].copy()
cand_tr["Source"] = "Traditional"
cand_tr["PredictorObj"] = "tr"

cand_dl = lb_dl[["model", "score_test"]].copy()
cand_dl["Source"] = "DeepLearning"
cand_dl["PredictorObj"] = "dl"

# --- FT candidate (manual accuracy using safe_binary_metrics + chunk logic) ---
# Get FT model name safely (works across versions)
ft_model_name = predictor_ft.leaderboard(silent=True)["model"].iloc[0]

# Use chunking only if test_df is large
chunk_threshold = 50000
if len(test_df) >= chunk_threshold:
    ft_metrics = safe_binary_metrics(
        predictor_ft,
        test_df,
        label="EnergyPoor",
        chunk_size=20000,
        pos_label=1
    )
    ft_acc = ft_metrics["Accuracy"]
else:
    # For small test sets, you can still use safe_binary_metrics if you want consistency
    ft_metrics = safe_binary_metrics(
        predictor_ft,
        test_df,
        label="EnergyPoor",
        chunk_size=20000,
        pos_label=1
    )
    ft_acc = ft_metrics["Accuracy"]

cand_ft = pd.DataFrame([{
    "model": ft_model_name,
    "score_test": ft_acc,
    "Source": "FTTransformer",
    "PredictorObj": "ft"
}])

# --- Combine all candidates ---
candidates = pd.concat([cand_tr, cand_dl, cand_ft], ignore_index=True)

# Remove ensemble rows if any (safety)
candidates = candidates[~candidates["model"].str.contains("WeightedEnsemble", na=False)]

# --- Pick best overall by accuracy ---
best_row = candidates.sort_values("score_test", ascending=False).iloc[0]

best_model_name = best_row["model"]
best_source = best_row["Source"]

if best_row["PredictorObj"] == "tr":
    best_predictor = predictor_tr
elif best_row["PredictorObj"] == "dl":
    best_predictor = predictor_dl
else:
    best_predictor = predictor_ft

print("Best overall model:", best_model_name)
print("From:", best_source)
print("Test Accuracy:", best_row["score_test"])


In [ ]:
subsample_size = min(10000, len(test_df))  # good for speed + consistent with report

fi_best = best_predictor.feature_importance(
    data=test_df,
    model=best_model_name,
    subsample_size=subsample_size,
    num_shuffle_sets=5
)

#fi_best

# Convert index (feature names) to column
fi_formatted = (
    fi_best
    .reset_index()
    .rename(columns={
        "index": "Feature",
        "importance": "Importance",
        "stddev": "Std Dev",
        "p_value": "P-value",
        "p99_high": "CI Upper (99%)",
        "p99_low": "CI Lower (99%)"
    })
)

# Add Country column
fi_formatted.insert(0, "Country", "India_Wealth_Inclusive")

# Reorder columns to match your screenshot
fi_formatted = fi_formatted[[
    "Country",
    "Feature",
    "Importance",
    "Std Dev",
    "P-value",
    "CI Lower (99%)",
    "CI Upper (99%)",
]]

# Round values for cleaner presentation
fi_formatted = fi_formatted.round({
    "Importance": 6,
    "Std Dev": 6,
    "P-value": 6,
    "CI Lower (99%)": 6,
    "CI Upper (99%)": 6,
})

fi_formatted

In [19]:
fi_formatted.to_csv("INDIA/RESULT/TableA5_india_feature_important_wealth_inclusive.csv", index=False)